In [ ]:
!nvidia-smi

Sun May 10 18:18:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install git+https://github.com/andreinechaev/nvcc4jupyter.git

  Cloning https://github.com/andreinechaev/nvcc4jupyter.git to /tmp/pip-req-build-rkizk8jf
  Running command git clone --filter=blob:none --quiet https://github.com/andreinechaev/nvcc4jupyter.git /tmp/pip-req-build-rkizk8jf
  Resolved https://github.com/andreinechaev/nvcc4jupyter.git to commit 28f872a2f99a1b201bcd0db14fdbc5a496b9bfd7
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for nvcc4jupyter: filename=nvcc4jupyter-1.2.1-py3-none-any.whl size=10741 sha256=90a829d54142cdbaf5703d6409de30ec0002408286cfbcd108dfdd84cf0ee4fe
  Stored in directory: /tmp/pip-ephem-wheel-cache-dl8v3hw3/wheels/7d/b9/66/459b9938664e6a93d1a85323ec52f7e51cd7265d253410a7d8
Successfully built nvcc4jupyter


In [ ]:
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmpdo42myht".


In [ ]:
%%writefile vector.cu

#include<iostream>
#include<cuda_runtime.h>
#include<chrono>
#include<stdlib.h>

using namespace std;
using namespace std::chrono;

#define N 1000000

// CUDA Kernel
__global__
void vectorAdd(int *A, int *B, int *C, int n)
{
    int tid = blockIdx.x * blockDim.x + threadIdx.x;

    if(tid < n)
    {
        C[tid] = A[tid] + B[tid];
    }
}

// Initialize Vector
void initialize(int *vector, int n)
{
    for(int i = 0; i < n; i++)
    {
        vector[i] = rand() % 10;
    }
}

// Sequential Addition
void sequentialAdd(int *A, int *B, int *C, int n)
{
    for(int i = 0; i < n; i++)
    {
        C[i] = A[i] + B[i];
    }
}

int main()
{
    int *A, *B, *C_seq, *C_parallel;

    size_t bytes = N * sizeof(int);

    // Host Memory
    A = new int[N];
    B = new int[N];
    C_seq = new int[N];
    C_parallel = new int[N];

    initialize(A, N);
    initialize(B, N);

    // Sequential Timing
    auto start_cpu = high_resolution_clock::now();

    sequentialAdd(A, B, C_seq, N);

    auto stop_cpu = high_resolution_clock::now();

    auto cpu_duration =
    duration_cast<microseconds>(stop_cpu - start_cpu);

    cout << "Sequential Time: "
         << cpu_duration.count()
         << " microseconds" << endl;

    // Device Memory
    int *d_A, *d_B, *d_C;

    cudaMalloc(&d_A, bytes);
    cudaMalloc(&d_B, bytes);
    cudaMalloc(&d_C, bytes);

    cudaMemcpy(d_A, A, bytes,
               cudaMemcpyHostToDevice);

    cudaMemcpy(d_B, B, bytes,
               cudaMemcpyHostToDevice);

    // CUDA Timing
    cudaEvent_t start_gpu, stop_gpu;

    cudaEventCreate(&start_gpu);
    cudaEventCreate(&stop_gpu);

    int threads = 256;
    int blocks = (N + threads - 1) / threads;

    cudaEventRecord(start_gpu);

    vectorAdd<<<blocks, threads>>>(d_A,
                                   d_B,
                                   d_C,
                                   N);

    cudaEventRecord(stop_gpu);

    cudaEventSynchronize(stop_gpu);

    float gpu_time = 0;

    cudaEventElapsedTime(&gpu_time,
                         start_gpu,
                         stop_gpu);

    cudaMemcpy(C_parallel,
               d_C,
               bytes,
               cudaMemcpyDeviceToHost);

    cout << "Parallel GPU Time: "
         << gpu_time
         << " ms" << endl;

    // Speedup
    float cpu_ms = cpu_duration.count() / 1000.0;

    cout << "Speedup: "
         << cpu_ms / gpu_time
         << endl;

    // Print First 10 Results
    cout << endl;
    cout << "First 10 Results:" << endl;

    for(int i = 0; i < 10; i++)
    {
        cout << C_parallel[i] << " ";
    }

    cout << endl;

    // Free Memory
    delete[] A;
    delete[] B;
    delete[] C_seq;
    delete[] C_parallel;

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}

Writing vector.cu


In [ ]:
!nvcc vector.cu -o vector

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./vector


Sequential Time: 4986 microseconds
Parallel GPU Time: 15.3204 ms
Speedup: 0.325449

First 10 Results:
12 9 11 8 4 10 12 5 10 1 


In [ ]:
%%writefile matrix.cu

#include<iostream>
#include<cuda_runtime.h>
#include<chrono>
#include<stdlib.h>

using namespace std;
using namespace std::chrono;

#define N 512

// CUDA Kernel
__global__
void matrixMultiply(int *A, int *B, int *C, int n)
{
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if(row < n && col < n)
    {
        int sum = 0;

        for(int k = 0; k < n; k++)
        {
            sum += A[row * n + k] *
                   B[k * n + col];
        }

        C[row * n + col] = sum;
    }
}

// Initialize Matrix
void initializeMatrix(int *matrix, int n)
{
    for(int i = 0; i < n * n; i++)
    {
        matrix[i] = rand() % 10;
    }
}

// Sequential Matrix Multiplication
void sequentialMultiply(int *A,
                        int *B,
                        int *C,
                        int n)
{
    for(int i = 0; i < n; i++)
    {
        for(int j = 0; j < n; j++)
        {
            int sum = 0;

            for(int k = 0; k < n; k++)
            {
                sum += A[i * n + k] *
                       B[k * n + j];
            }

            C[i * n + j] = sum;
        }
    }
}

int main()
{
    int size = N * N;
    size_t bytes = size * sizeof(int);

    int *A, *B, *C_seq, *C_parallel;

    // Host Memory
    A = new int[size];
    B = new int[size];
    C_seq = new int[size];
    C_parallel = new int[size];

    initializeMatrix(A, N);
    initializeMatrix(B, N);

    // Sequential Timing
    auto start_cpu = high_resolution_clock::now();

    sequentialMultiply(A, B, C_seq, N);

    auto stop_cpu = high_resolution_clock::now();

    auto cpu_duration =
    duration_cast<microseconds>(stop_cpu - start_cpu);

    cout << "Sequential Time: "
         << cpu_duration.count()
         << " microseconds" << endl;

    // Device Memory
    int *d_A, *d_B, *d_C;

    cudaMalloc(&d_A, bytes);
    cudaMalloc(&d_B, bytes);
    cudaMalloc(&d_C, bytes);

    cudaMemcpy(d_A, A, bytes,
               cudaMemcpyHostToDevice);

    cudaMemcpy(d_B, B, bytes,
               cudaMemcpyHostToDevice);

    // CUDA Timing
    cudaEvent_t start_gpu, stop_gpu;

    cudaEventCreate(&start_gpu);
    cudaEventCreate(&stop_gpu);

    dim3 threads(16,16);

    dim3 blocks((N + threads.x - 1) / threads.x,
                (N + threads.y - 1) / threads.y);

    cudaEventRecord(start_gpu);

    matrixMultiply<<<blocks, threads>>>(d_A,
                                        d_B,
                                        d_C,
                                        N);

    cudaEventRecord(stop_gpu);

    cudaEventSynchronize(stop_gpu);

    float gpu_time = 0;

    cudaEventElapsedTime(&gpu_time,
                         start_gpu,
                         stop_gpu);

    cudaMemcpy(C_parallel,
               d_C,
               bytes,
               cudaMemcpyDeviceToHost);

    cout << "Parallel GPU Time: "
         << gpu_time
         << " ms" << endl;

    // Speedup
    float cpu_ms = cpu_duration.count() / 1000.0;

    cout << "Speedup: "
         << cpu_ms / gpu_time
         << endl;

    // Print First 4x4 Result Matrix
    cout << endl;
    cout << "First 4x4 Result Matrix:" << endl;

    for(int i = 0; i < 4; i++)
    {
        for(int j = 0; j < 4; j++)
        {
            cout << C_parallel[i * N + j] << " ";
        }

        cout << endl;
    }

    // Free Memory
    delete[] A;
    delete[] B;
    delete[] C_seq;
    delete[] C_parallel;

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}

Writing matrix.cu


In [ ]:
!nvcc matrix.cu -o matrix

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./matrix

Sequential Time: 761846 microseconds
Parallel GPU Time: 22.439 ms
Speedup: 33.9519

First 4x4 Result Matrix:
11057 10652 10663 10558 
10651 10596 10164 10047 
10817 10505 10400 10177 
10208 9973 10062 10217 
